# 📊 Análise Exploratória de Imagens de Obra — IPOS

**Profª M.Sc. Adriana Rolim** — IPOS Pós-Graduação
*Disciplina: Visão Computacional Aplicada à Construção Civil*

---

## 🎯 Objetivo

Antes de treinar qualquer modelo de Visão Computacional (U-Net, YOLO, etc), precisamos **entender o dataset**. Imagens de obra têm desafios específicos: variação de iluminação, ângulos imprecisos, ruído de poeira, dimensões inconsistentes entre câmeras.

Este notebook executa duas análises:

1. **Análise exploratória do dataset** — quantas imagens, tamanhos, formatos, proporções
2. **Diagnóstico por imagem** — 8 verificações que indicam *quais pré-processamentos cada foto precisa*

E ao final, gera um **relatório HTML interativo** que pode ser anexado a parecer técnico ou compartilhado com a equipe.

---

## 📁 Pasta de entrada

Pasta no Google Drive: `fotos_obra/`
🔗 https://drive.google.com/drive/folders/1oGblbMJE9rCZsk8jUOy56XrApuv6BNpf

## 0️⃣ Setup — Montar Google Drive

Esta célula só funciona no **Google Colab**. Vai pedir autorização para acessar seu Drive na primeira execução.

In [ ]:
# Monta o Google Drive no Colab
from google.colab import drive
drive.mount('/content/drive')

# Caminho da pasta de fotos
PASTA_FOTOS = '/content/drive/MyDrive/fotos_obra'

# Confirmação
import os
if os.path.exists(PASTA_FOTOS):
    n_arquivos = len([f for f in os.listdir(PASTA_FOTOS) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f"✅ Pasta encontrada: {PASTA_FOTOS}")
    print(f"   {n_arquivos} imagens detectadas")
else:
    print(f"⚠️ Pasta não encontrada: {PASTA_FOTOS}")
    print("   Verifique se a pasta 'fotos_obra' está na raiz do seu MyDrive")
    print("   Se estiver em subpasta, ajuste PASTA_FOTOS acima")

---
## 1️⃣ Importações e Configuração

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from collections import Counter
from datetime import datetime
import base64
from pathlib import Path
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 90

print("✅ Bibliotecas carregadas")
print(f"   OpenCV: {cv2.__version__}")
print(f"   Pandas: {pd.__version__}")

---
## 2️⃣ Análise Exploratória do Dataset

A primeira pergunta antes de qualquer projeto de visão computacional é:

> *"Como são as imagens que tenho?"*

A função `analisar_dataset()` percorre a pasta e coleta:
- **Quantidade** de imagens válidas
- **Dimensões** (altura e largura: mínima, máxima, média)
- **Canais** de cor (3 = RGB, 4 = RGBA, 1 = grayscale)
- **Formatos** (JPG, PNG, JPEG)
- **Proporções** (largura/altura — ex: 1.78 = 16:9)

Por que isso importa: modelos de IA esperam dimensões padronizadas. Se há fotos 4000×3000 e outras 800×600 misturadas, o pipeline precisa redimensionar. Saber a distribuição evita surpresas.

In [ ]:
def analisar_dataset(diretorio):
    """
    Realiza análise exploratória básica do dataset de imagens.

    Parâmetros:
        diretorio (str): Caminho para o diretório com as imagens.

    Retorno:
        dict: Informações estatísticas como tamanho médio, formatos e proporções.
        int: Total de imagens válidas analisadas.
        dict: Dados brutos (para visualizações posteriores).
    """
    alturas, larguras, canais, formatos, proporcoes = [], [], [], [], []
    nomes_validos = []
    total_imagens = 0

    for nome in sorted(os.listdir(diretorio)):
        if nome.lower().endswith(('.jpg', '.jpeg', '.png')):
            img = cv2.imread(os.path.join(diretorio, nome))
            if img is None:
                continue
            h, w, c = img.shape
            alturas.append(h)
            larguras.append(w)
            canais.append(c)
            formatos.append(nome.split('.')[-1].lower())
            proporcoes.append(round(w / h, 2))
            nomes_validos.append(nome)
            total_imagens += 1

    if total_imagens == 0:
        return "Nenhuma imagem válida encontrada no diretório.", None, None

    dados = {
        'Total de Imagens': total_imagens,
        'Altura Média': int(np.mean(alturas)),
        'Largura Média': int(np.mean(larguras)),
        'Altura Mínima': min(alturas),
        'Altura Máxima': max(alturas),
        'Largura Mínima': min(larguras),
        'Largura Máxima': max(larguras),
        'Canais Detectados': dict(Counter(canais)),
        'Formatos de Imagem': dict(Counter(formatos)),
        'Proporções (L/A)': dict(Counter(proporcoes))
    }

    brutos = {
        'alturas': alturas, 'larguras': larguras,
        'canais': canais, 'formatos': formatos,
        'proporcoes': proporcoes, 'nomes': nomes_validos,
    }

    return dados, total_imagens, brutos


# Executa
info_dataset, total, brutos = analisar_dataset(PASTA_FOTOS)

if isinstance(info_dataset, str):
    print(info_dataset)
else:
    print("="*50)
    print("📊 INFORMAÇÕES DO DATASET")
    print("="*50)
    for k, v in info_dataset.items():
        print(f"  {k:25} {v}")
    print("="*50)

### 📈 Visualização das Dimensões

Tabela é boa para auditoria, gráfico é melhor para entender. Vamos plotar:

- **Histograma de larguras e alturas** — para ver se há dois grupos de câmeras
- **Scatter largura × altura** — para identificar outliers
- **Distribuição de proporções** — fotos quadradas? 16:9? 4:3?

In [ ]:
# Só executa se há dados
if brutos:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Histograma de larguras e alturas
    axes[0].hist(brutos['larguras'], bins=15, alpha=0.6, label='Largura', color='#3498db')
    axes[0].hist(brutos['alturas'],  bins=15, alpha=0.6, label='Altura',  color='#e74c3c')
    axes[0].set_xlabel('Pixels')
    axes[0].set_ylabel('Nº de imagens')
    axes[0].set_title('Distribuição de Dimensões')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Scatter largura × altura
    axes[1].scatter(brutos['larguras'], brutos['alturas'], alpha=0.6, s=30, color='#27ae60')
    axes[1].set_xlabel('Largura (px)')
    axes[1].set_ylabel('Altura (px)')
    axes[1].set_title('Largura × Altura (cada ponto = 1 foto)')
    axes[1].grid(alpha=0.3)

    # Proporções
    prop_counter = Counter(brutos['proporcoes'])
    props_sorted = sorted(prop_counter.items())
    axes[2].bar([str(p) for p,_ in props_sorted], [c for _,c in props_sorted], color='#f39c12')
    axes[2].set_xlabel('Proporção (L/A)')
    axes[2].set_ylabel('Nº de imagens')
    axes[2].set_title('Distribuição de Proporções')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Interpretação automática
    print("\n🔍 INTERPRETAÇÃO:")
    if len(set(brutos['larguras'])) <= 2:
        print(f"  ✓ Larguras consistentes ({len(set(brutos['larguras']))} valor(es) distinto(s)) — pipeline simples")
    else:
        print(f"  ⚠ {len(set(brutos['larguras']))} larguras distintas — vai precisar redimensionar")

    if len(set(brutos['proporcoes'])) == 1:
        print(f"  ✓ Todas as fotos têm a mesma proporção ({brutos['proporcoes'][0]})")
    else:
        print(f"  ⚠ {len(set(brutos['proporcoes']))} proporções diferentes — cuidado ao recortar")

### 👀 Amostra Visual de 6 Fotos

Antes de qualquer análise automatizada, **olhar** algumas fotos. Isso revela problemas que estatística não pega: fotos tortas, lente embaçada, dedo na frente da câmera, etc.

In [ ]:
if brutos and len(brutos['nomes']) > 0:
    # Pega até 6 amostras aleatórias (com seed para reprodutibilidade)
    np.random.seed(42)
    n_amostras = min(6, len(brutos['nomes']))
    idx_amostras = np.random.choice(len(brutos['nomes']), n_amostras, replace=False)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, idx in enumerate(idx_amostras):
        nome = brutos['nomes'][idx]
        caminho = os.path.join(PASTA_FOTOS, nome)
        img = cv2.imread(caminho)
        if img is not None:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[i].set_title(f'{nome}\n{brutos["larguras"][idx]}×{brutos["alturas"][idx]}', fontsize=9)
            axes[i].axis('off')

    # Esconde subplots vazios
    for j in range(n_amostras, 6):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()
    print(f"📸 {n_amostras} amostras aleatórias do dataset.")

---
## 3️⃣ Verificações de Pré-processamento

Para cada imagem, aplicamos **8 verificações** que indicam **quais pré-processamentos** ela vai precisar. Cada verificação retorna `True` (precisa) ou `False` (não precisa).

### Por que essas 8?

| Verificação | O que detecta | Quando dá True, o que fazer |
|---|---|---|
| **Redimensionamento** | Foto maior que 800×800 | `cv2.resize()` para padrão do modelo |
| **Redução de Ruído** | Alto desvio padrão de pixels | Aplicar `cv2.GaussianBlur` ou bilateral |
| **Aumento de Contraste** | Canal Y com baixo desvio (<30) | CLAHE (`cv2.createCLAHE`) ou equalização |
| **Detecção de Bordas** | Poucas bordas detectadas pelo Canny | Verificar foco; pode estar desfocada |
| **Conversão Espaço Cor** | Confirma que é RGB de 3 canais | Converter de RGBA/grayscale se necessário |
| **Aumento de Dados** | Foto pequena (<500×500) | Data augmentation para treino |
| **Remoção de Artefatos** | Diferença alta entre original e suavizada | Median filter, denoising |
| **Subtração de Fundo** | Fundo com alta variação | Aplicar masking ou pré-segmentação |

Vamos definir cada uma com o seu critério.

In [ ]:
# ============================
# Verificações de Pré-processamento (do script original)
# ============================

def verificar_redimensionamento(img, tamanho_alvo=(800, 800)):
    """Verifica se a imagem excede o tamanho recomendado."""
    h, w = img.shape[:2]
    return h > tamanho_alvo[0] or w > tamanho_alvo[1]

def verificar_reducao_ruido(img):
    """Estima a presença de ruído com base no desvio padrão dos pixels."""
    return np.std(img) > 50

def verificar_aumento_contraste(img):
    """Avalia se o canal de luminância tem contraste suficiente."""
    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    y = ycrcb[:, :, 0]
    return y.std() < 30

def verificar_bordas(img):
    """Verifica se há bordas suficientes usando o detector Canny."""
    edges = cv2.Canny(img, 100, 200)
    borda_ratio = np.sum(edges > 0) / edges.size
    return borda_ratio < 0.01

def verificar_conversao_espaco_cor(img):
    """Verifica se a imagem está em RGB (3 canais)."""
    return img.shape[2] == 3

def verificar_aumento_dados(img):
    """Detecta se a imagem é pequena e pode se beneficiar de data augmentation."""
    h, w = img.shape[:2]
    return h * w < 500 * 500

def verificar_remocao_artefatos(img):
    """Estima presença de artefatos visuais com base na diferença da imagem suavizada."""
    blur = cv2.medianBlur(img, 5)
    diff = cv2.absdiff(img, blur)
    artefatos = np.mean(diff)
    return artefatos > 10

def verificar_subtracao_fundo(img):
    """Analisa a uniformidade do fundo no canto superior da imagem."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bg_std = np.std(gray[:50, :50])
    return bg_std > 15


print("✅ 8 funções de verificação definidas")

---
## 4️⃣ Aplicação das Verificações em Todas as Imagens

Roda as 8 verificações em cada imagem e monta um DataFrame consolidado.

In [ ]:
def analisar_imagens(diretorio):
    """
    Aplica as 8 verificações de pré-processamento em cada imagem.

    Parâmetros:
        diretorio (str): Caminho para o diretório com imagens.

    Retorno:
        pd.DataFrame: Tabela com recomendações por imagem.
    """
    resultados = []
    for nome_arquivo in sorted(os.listdir(diretorio)):
        if nome_arquivo.lower().endswith(('.jpg', '.jpeg', '.png')):
            caminho = os.path.join(diretorio, nome_arquivo)
            img = cv2.imread(caminho)
            if img is None:
                continue
            analise = {
                'Imagem': nome_arquivo,
                'Redimensionamento':       verificar_redimensionamento(img),
                'Redução de Ruído':        verificar_reducao_ruido(img),
                'Aumento de Contraste':    verificar_aumento_contraste(img),
                'Detecção de Bordas':      verificar_bordas(img),
                'Conversão Espaço de Cor': verificar_conversao_espaco_cor(img),
                'Aumento de Dados':        verificar_aumento_dados(img),
                'Remoção de Artefatos':    verificar_remocao_artefatos(img),
                'Subtração de Fundo':      verificar_subtracao_fundo(img)
            }
            resultados.append(analise)
    return pd.DataFrame(resultados)


# Executa
print("🔍 Analisando todas as imagens... (pode levar alguns segundos)")
df = analisar_imagens(PASTA_FOTOS)
print(f"\n✅ {len(df)} imagens analisadas\n")

# Mostra as primeiras linhas
df.head(10)

### 📊 Visão Consolidada — Quantas imagens precisam de cada pré-processamento?

Em vez de olhar linha por linha, fazemos o **resumo agregado**: quantas % das fotos precisam de cada tratamento.

In [ ]:
# Conta True por coluna (ignora 'Imagem')
cols_verificacao = [c for c in df.columns if c != 'Imagem']
resumo = df[cols_verificacao].sum().sort_values(ascending=True)

# Gráfico horizontal
fig, ax = plt.subplots(figsize=(10, 5))
cores = ['#27ae60' if v < len(df)*0.3 else '#f39c12' if v < len(df)*0.6 else '#c0392b'
         for v in resumo.values]
ax.barh(resumo.index, resumo.values, color=cores)
ax.set_xlabel('Nº de imagens que precisam')
ax.set_title(f'Pré-processamentos necessários (total: {len(df)} imagens)')
ax.grid(axis='x', alpha=0.3)

# Anota porcentagem
for i, v in enumerate(resumo.values):
    ax.text(v + 0.1, i, f' {v} ({100*v/len(df):.0f}%)', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📋 Tabela resumida:")
resumo_df = pd.DataFrame({
    'Verificação': resumo.index,
    'Nº imagens': resumo.values,
    '% do dataset': [f'{100*v/len(df):.0f}%' for v in resumo.values]
})
print(resumo_df.to_string(index=False))

### 🗺️ Mapa de Calor — Quais imagens precisam de quê?

Visão por imagem (linhas) × verificação (colunas). Útil para identificar **imagens problemáticas** (linhas com muitas marcações) que talvez devam ser descartadas.

In [ ]:
# Converte DataFrame para matriz numérica (True=1, False=0)
matriz = df[cols_verificacao].astype(int).values

fig, ax = plt.subplots(figsize=(12, max(4, len(df)*0.25)))
im = ax.imshow(matriz, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)

# Eixos
ax.set_xticks(range(len(cols_verificacao)))
ax.set_xticklabels(cols_verificacao, rotation=30, ha='right')
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['Imagem'].values, fontsize=8)

# Texto nas células
for i in range(len(df)):
    for j in range(len(cols_verificacao)):
        v = matriz[i, j]
        ax.text(j, i, '●' if v else '·',
                ha='center', va='center',
                color='white' if v else 'gray', fontsize=10)

ax.set_title(f'Mapa de pré-processamentos necessários\n(● = sim, · = não)')
plt.tight_layout()
plt.show()

# Identifica imagens problemáticas
df_score = df.copy()
df_score['Total_Marcacoes'] = df[cols_verificacao].sum(axis=1)
problematicas = df_score.nlargest(3, 'Total_Marcacoes')[['Imagem', 'Total_Marcacoes']]
print("\n⚠️ 3 imagens com MAIS pré-processamentos necessários:")
print(problematicas.to_string(index=False))

---
## 5️⃣ Salvar CSV (output original do script)

In [ ]:
# Mantém o output do script original
csv_path = 'relatorio_preprocessamento.csv'
df.to_csv(csv_path, index=False)
print(f"✅ CSV salvo: {csv_path}")
print(f"   Tamanho: {os.path.getsize(csv_path)} bytes")

# Se estiver no Colab, oferece download
try:
    from google.colab import files
    print("\n💡 Para baixar o CSV, descomente a linha abaixo:")
    print("# files.download('relatorio_preprocessamento.csv')")
except ImportError:
    pass

---
## 6️⃣ Relatório HTML Didático

Esta é a etapa **adicionada** ao script original. Vamos gerar um **arquivo HTML standalone** que:

- Apresenta as informações do dataset com cards visuais
- Explica **o significado** de cada uma das 8 verificações
- Mostra o **que fazer** quando cada verificação dá True
- Inclui o mapa de calor embutido como imagem
- Lista as imagens problemáticas com ações sugeridas

Esse HTML pode ser:
- 📧 Compartilhado com a equipe da obra
- 📋 Anexado ao parecer técnico
- 🌐 Aberto em qualquer browser sem servidor

In [ ]:
def fig_para_base64(fig):
    """Converte uma figura matplotlib em data URI base64."""
    import io
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    return f"data:image/png;base64,{base64.b64encode(buf.read()).decode()}"


def gerar_relatorio_html(saida, info_dataset, brutos, df, pasta_fotos):
    """Gera relatório HTML didático com os resultados da análise."""

    # ===== Regera as figuras para embutir =====
    # Fig 1: distribuição de dimensões
    fig1, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(brutos['larguras'], bins=15, alpha=0.6, label='Largura', color='#3498db')
    axes[0].hist(brutos['alturas'], bins=15, alpha=0.6, label='Altura', color='#e74c3c')
    axes[0].set_xlabel('Pixels'); axes[0].set_ylabel('Nº de imagens')
    axes[0].set_title('Distribuição de Dimensões'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].scatter(brutos['larguras'], brutos['alturas'], alpha=0.6, s=30, color='#27ae60')
    axes[1].set_xlabel('Largura (px)'); axes[1].set_ylabel('Altura (px)')
    axes[1].set_title('Largura × Altura'); axes[1].grid(alpha=0.3)
    prop_counter = Counter(brutos['proporcoes'])
    props_sorted = sorted(prop_counter.items())
    axes[2].bar([str(p) for p,_ in props_sorted], [c for _,c in props_sorted], color='#f39c12')
    axes[2].set_xlabel('Proporção (L/A)'); axes[2].set_ylabel('Nº')
    axes[2].set_title('Proporções'); axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    img_dims_b64 = fig_para_base64(fig1)
    plt.close(fig1)

    # Fig 2: barras horizontais resumo
    cols_v = [c for c in df.columns if c != 'Imagem']
    resumo = df[cols_v].sum().sort_values(ascending=True)
    fig2, ax = plt.subplots(figsize=(10, 5))
    cores = ['#27ae60' if v < len(df)*0.3 else '#f39c12' if v < len(df)*0.6 else '#c0392b' for v in resumo.values]
    ax.barh(resumo.index, resumo.values, color=cores)
    ax.set_xlabel('Nº de imagens que precisam')
    ax.set_title(f'Pré-processamentos necessários (n={len(df)})')
    ax.grid(axis='x', alpha=0.3)
    for i, v in enumerate(resumo.values):
        ax.text(v + 0.1, i, f' {v} ({100*v/len(df):.0f}%)', va='center', fontsize=9)
    plt.tight_layout()
    img_resumo_b64 = fig_para_base64(fig2)
    plt.close(fig2)

    # Fig 3: mapa de calor
    matriz = df[cols_v].astype(int).values
    fig3, ax = plt.subplots(figsize=(11, max(4, len(df)*0.25)))
    ax.imshow(matriz, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(cols_v))); ax.set_xticklabels(cols_v, rotation=30, ha='right')
    ax.set_yticks(range(len(df))); ax.set_yticklabels(df['Imagem'].values, fontsize=8)
    for i in range(len(df)):
        for j in range(len(cols_v)):
            v = matriz[i, j]
            ax.text(j, i, '●' if v else '·',
                    ha='center', va='center',
                    color='white' if v else 'gray', fontsize=10)
    ax.set_title('Mapa de pré-processamentos por imagem')
    plt.tight_layout()
    img_mapa_b64 = fig_para_base64(fig3)
    plt.close(fig3)

    # ===== Conteúdo do HTML =====
    # Glossário didático de cada verificação
    GLOSSARIO = {
        'Redimensionamento': {
            'desc': 'Imagem maior que 800×800 pixels. Pode ser pesada para o modelo.',
            'acao': 'Aplicar cv2.resize() para o tamanho de entrada do modelo (ex: 256×256 ou 512×512).',
            'icone': '📐',
        },
        'Redução de Ruído': {
            'desc': 'Pixels com alta variação (ruído de sensor, poeira, granulação).',
            'acao': 'Aplicar cv2.GaussianBlur ou cv2.bilateralFilter antes de outras operações.',
            'icone': '🔇',
        },
        'Aumento de Contraste': {
            'desc': 'Canal de luminância pouco variado (foto "lavada", sem contraste).',
            'acao': 'Aplicar CLAHE (cv2.createCLAHE) no canal Y do espaço YCrCb.',
            'icone': '🌗',
        },
        'Detecção de Bordas': {
            'desc': 'Canny encontrou poucas bordas (<1%). Pode estar desfocada ou ser superfície lisa.',
            'acao': 'Verificar foco da foto. Se OK, prosseguir — é apenas baixa estrutura visual.',
            'icone': '✏️',
        },
        'Conversão Espaço de Cor': {
            'desc': 'Confirmação de que a imagem está em RGB (3 canais).',
            'acao': 'Se False: converter de grayscale (cv2.COLOR_GRAY2BGR) ou RGBA.',
            'icone': '🎨',
        },
        'Aumento de Dados': {
            'desc': 'Imagem pequena (<500×500). Pode prejudicar generalização do modelo.',
            'acao': 'Aplicar data augmentation: rotação, flip, crop, jitter de cor.',
            'icone': '🔁',
        },
        'Remoção de Artefatos': {
            'desc': 'Diferença alta entre imagem original e suavizada. Indica artefatos JPEG, riscos, manchas.',
            'acao': 'Aplicar cv2.medianBlur ou cv2.fastNlMeansDenoising.',
            'icone': '🧹',
        },
        'Subtração de Fundo': {
            'desc': 'Fundo (canto superior) com alta variação. Dificulta segmentação por subtração.',
            'acao': 'Aplicar segmentação semântica em vez de subtração simples (ex: U-Net).',
            'icone': '🪟',
        },
    }

    # Cards do glossário
    cards_glossario = ''
    for nome in cols_v:
        info = GLOSSARIO.get(nome, {'desc': '', 'acao': '', 'icone': '🔍'})
        n_true = int(df[nome].sum())
        pct = 100 * n_true / len(df)
        cor = '#27ae60' if pct < 30 else '#f39c12' if pct < 60 else '#c0392b'
        cards_glossario += f"""
        <div class="card-glossario">
          <div class="card-titulo">{info['icone']} {nome}</div>
          <div class="card-stat" style="color:{cor}">{n_true}/{len(df)} ({pct:.0f}%)</div>
          <div class="card-desc"><b>O que é:</b> {info['desc']}</div>
          <div class="card-acao"><b>O que fazer:</b> {info['acao']}</div>
        </div>"""

    # Tabela de imagens problemáticas
    df_score = df.copy()
    df_score['Total'] = df[cols_v].sum(axis=1)
    problematicas = df_score.nlargest(min(5, len(df)), 'Total')

    linhas_problem = ''
    for _, row in problematicas.iterrows():
        marcadas = [c for c in cols_v if row[c]]
        marcadas_str = ', '.join(marcadas) if marcadas else '—'
        linhas_problem += f"""<tr>
        <td><b>{row['Imagem']}</b></td>
        <td style="text-align:center"><span class="badge-num">{row['Total']}</span></td>
        <td>{marcadas_str}</td>
        </tr>"""

    # Cards principais do dataset
    cards_dataset = f"""
    <div class="card-info">
      <div class="card-info-titulo">Total de Imagens</div>
      <div class="card-info-valor">{info_dataset['Total de Imagens']}</div>
    </div>
    <div class="card-info">
      <div class="card-info-titulo">Dimensão Média</div>
      <div class="card-info-valor">{info_dataset['Largura Média']}×{info_dataset['Altura Média']}</div>
      <div class="card-info-sub">L × A em pixels</div>
    </div>
    <div class="card-info">
      <div class="card-info-titulo">Faixa de Largura</div>
      <div class="card-info-valor">{info_dataset['Largura Mínima']}–{info_dataset['Largura Máxima']}</div>
      <div class="card-info-sub">mín–máx em pixels</div>
    </div>
    <div class="card-info">
      <div class="card-info-titulo">Formatos</div>
      <div class="card-info-valor" style="font-size:18px">{', '.join(info_dataset['Formatos de Imagem'].keys())}</div>
    </div>"""

    # ===== Monta HTML =====
    html = """<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>Relatorio EDA - Imagens de Obra</title>
<style>
* {box-sizing:border-box;margin:0;padding:0}
body {font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;background:#f4f6fa;color:#2c3e50;line-height:1.6}
header {background:linear-gradient(135deg,#1f3864 0%,#2e5cb8 100%);color:white;padding:32px;box-shadow:0 2px 8px rgba(0,0,0,.1)}
header h1 {font-size:24px;margin-bottom:6px}
header .sub {opacity:.85;font-size:14px}
.container {max-width:1200px;margin:0 auto;padding:32px}
.secao {margin-bottom:36px}
.secao h2 {font-size:18px;margin-bottom:16px;color:#1f3864;border-left:4px solid #2e5cb8;padding-left:14px}
.secao p {margin-bottom:12px;color:#4a5568}
.grid-info {display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:24px}
.card-info {background:white;border-radius:8px;padding:20px;box-shadow:0 1px 3px rgba(0,0,0,.08);text-align:center}
.card-info-titulo {font-size:11px;text-transform:uppercase;letter-spacing:.5px;color:#7f8c8d;margin-bottom:8px;font-weight:600}
.card-info-valor {font-size:28px;font-weight:700;color:#2c3e50}
.card-info-sub {font-size:11px;color:#95a5a6;margin-top:4px}
.figura {background:white;border-radius:8px;padding:20px;box-shadow:0 1px 3px rgba(0,0,0,.08);text-align:center}
.figura img {max-width:100%;height:auto}
.grid-glossario {display:grid;grid-template-columns:repeat(2,1fr);gap:16px}
.card-glossario {background:white;border-radius:8px;padding:18px;box-shadow:0 1px 3px rgba(0,0,0,.08)}
.card-titulo {font-size:14px;font-weight:600;color:#1f3864;margin-bottom:6px}
.card-stat {font-size:22px;font-weight:700;margin-bottom:10px}
.card-desc, .card-acao {font-size:13px;margin-bottom:6px;color:#4a5568}
.card-desc b, .card-acao b {color:#2c3e50}
table {width:100%;border-collapse:collapse;font-size:13px;background:white;border-radius:8px;overflow:hidden;box-shadow:0 1px 3px rgba(0,0,0,.08)}
th {background:#1f3864;color:white;padding:10px;text-align:left;font-weight:600;font-size:12px;text-transform:uppercase;letter-spacing:.3px}
td {padding:10px;border-bottom:1px solid #ecf0f1}
tr:hover td {background:#f8f9fc}
.badge-num {display:inline-block;background:#c0392b;color:white;padding:2px 8px;border-radius:10px;font-size:11px;font-weight:700}
.aviso {background:#e8f4fd;border-left:4px solid #3498db;padding:14px 18px;margin:18px 0;border-radius:4px;font-size:13px}
.aviso b {color:#1f3864}
@media(max-width:900px){.grid-info{grid-template-columns:repeat(2,1fr)}.grid-glossario{grid-template-columns:1fr}}
footer {text-align:center;padding:24px;color:#95a5a6;font-size:12px;border-top:1px solid #ecf0f1;margin-top:32px}
</style>
</head>
<body>
<header>
<h1>📊 Relatorio de Analise Exploratoria - Imagens de Obra</h1>
<div class="sub">IPOS Pos-Graduacao | Profa M.Sc. Adriana Rolim | Emitido em __DATA__</div>
</header>
<div class="container">

<div class="secao">
<h2>1. Sobre Este Relatorio</h2>
<p>Esta analise foi realizada sobre o conjunto de <b>__TOTAL__ fotografias</b>
armazenadas na pasta <code>__PASTA__</code>. O objetivo e diagnosticar
<b>quais pre-processamentos</b> serao necessarios antes de usar essas imagens
em um modelo de visao computacional (deteccao de patologias, segmentacao de
elementos construtivos, etc).</p>
<div class="aviso">
<b>Como ler este relatorio:</b> A Secao 2 mostra os numeros gerais do dataset.
A Secao 3 explica cada uma das 8 verificacoes e mostra <b>quantas fotos</b>
precisam de cada tratamento. A Secao 4 lista as fotos mais problematicas.
</div>
</div>

<div class="secao">
<h2>2. Visao Geral do Dataset</h2>
<div class="grid-info">__CARDS_DATASET__</div>
<div class="figura">__IMG_DIMS__</div>
</div>

<div class="secao">
<h2>3. Diagnostico de Pre-processamento</h2>
<p>A barra abaixo mostra <b>quantas imagens</b> apresentaram a necessidade
de cada pre-processamento. Cada cartao explica o que e e o que fazer.</p>
<div class="figura">__IMG_RESUMO__</div>
<div class="grid-glossario" style="margin-top:20px">__CARDS_GLOSSARIO__</div>
</div>

<div class="secao">
<h2>4. Mapa Detalhado por Imagem</h2>
<p>Cada linha e uma foto; cada coluna uma verificacao. <b>Bola = precisa</b>;
<b>ponto = nao precisa</b>. Linhas com muitas bolas merecem revisao manual.</p>
<div class="figura">__IMG_MAPA__</div>
</div>

<div class="secao">
<h2>5. Imagens com Mais Marcacoes (revisar primeiro)</h2>
<p>Sao as imagens que precisarao do maior numero de pre-processamentos.
Antes de aplicar a sequencia automatica, vale conferir manualmente se
nao e melhor descarta-las ou re-fotografar.</p>
<table>
<thead><tr><th>Imagem</th><th style="text-align:center">N. de marcacoes</th><th>Pre-processamentos necessarios</th></tr></thead>
<tbody>__LINHAS_PROBLEM__</tbody>
</table>
</div>

<div class="secao">
<h2>6. Proximos Passos Recomendados</h2>
<p>Com base nesta analise, a sequencia de pipeline sugerida e:</p>
<ol style="margin-left:24px;line-height:2">
<li><b>Padronizar dimensoes</b> (se Redimensionamento marcou alto): <code>cv2.resize(img, (512, 512))</code></li>
<li><b>Converter espaco de cor</b> se necessario: garantir 3 canais BGR</li>
<li><b>Tratar ruido e artefatos</b> (Reducao de Ruido + Remocao de Artefatos): <code>cv2.bilateralFilter</code> ou <code>cv2.fastNlMeansDenoisingColored</code></li>
<li><b>Realcar contraste</b> (Aumento de Contraste): CLAHE no canal Y</li>
<li><b>Data augmentation</b> se Aumento de Dados marcou alto: <code>albumentations.Compose</code></li>
<li><b>Segmentacao avancada</b> em vez de subtracao simples (Subtracao de Fundo): U-Net ou Mask R-CNN</li>
</ol>
<div class="aviso" style="background:#fff8e1;border-left-color:#f39c12">
<b>Atencao:</b> as verificacoes deste relatorio sao <b>heuristicas baseadas
em estatistica de pixels</b>. Elas indicam <i>tendencia</i>, nao verdade
absoluta. Antes de aplicar transformacoes em lote, valide com algumas
amostras manuais.
</div>
</div>

</div>
<footer>
Gerado automaticamente | Analise Exploratoria de Imagens de Obra<br>
IPOS Pos-Graduacao - Profa M.Sc. Adriana Rolim
</footer>
</body>
</html>"""

    subs = {
        '__DATA__': datetime.now().strftime('%d/%m/%Y %H:%M'),
        '__TOTAL__': str(info_dataset['Total de Imagens']),
        '__PASTA__': pasta_fotos,
        '__CARDS_DATASET__': cards_dataset,
        '__IMG_DIMS__': f'<img src="{img_dims_b64}" alt="Distribuicao de dimensoes">',
        '__IMG_RESUMO__': f'<img src="{img_resumo_b64}" alt="Resumo de pre-processamentos">',
        '__IMG_MAPA__': f'<img src="{img_mapa_b64}" alt="Mapa detalhado">',
        '__CARDS_GLOSSARIO__': cards_glossario,
        '__LINHAS_PROBLEM__': linhas_problem,
    }
    for k, v in subs.items():
        html = html.replace(k, str(v))

    Path(saida).write_text(html, encoding='utf-8')
    return saida


# Gera o relatório
html_path = 'relatorio_eda_obra.html'
gerar_relatorio_html(html_path, info_dataset, brutos, df, PASTA_FOTOS)
print(f"✅ Relatório HTML gerado: {html_path}")
print(f"   Tamanho: {os.path.getsize(html_path):,} bytes".replace(',', '.'))

# No Colab, oferece download
try:
    from google.colab import files
    print("\n💡 Para baixar, descomente a linha abaixo:")
    print(f"# files.download('{html_path}')")
except ImportError:
    pass

---
## ✅ Resumo dos Outputs Gerados

| Arquivo | Conteúdo | Uso |
|---|---|---|
| `relatorio_preprocessamento.csv` | Tabela com 8 verificações por imagem | Análise tabular, importar no Excel/Power BI |
| `relatorio_eda_obra.html` | Relatório visual standalone com glossário e ações | Compartilhar com equipe, anexar a parecer |

### Como baixar no Colab

```python
from google.colab import files
files.download('relatorio_preprocessamento.csv')
files.download('relatorio_eda_obra.html')
```

### Próximos passos sugeridos

1. **Abrir o HTML** num navegador para apresentação visual
2. **Decidir** quais imagens descartar com base na Seção 5 do relatório
3. **Construir o pipeline de pré-processamento** seguindo a ordem recomendada na Seção 6
4. **Para fotos novas**, rodar este notebook de novo antes de incluir no dataset de treino

---

> **Profª M.Sc. Adriana Rolim** — IPOS Pós-Graduação
> *Análise Exploratória de Imagens de Obra*